# Data Stats

Computes the summary statistics described in `data/README.md`.

In [19]:
import json
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path(".")  # run from data/
TRANSCRIPTS = DATA_DIR / "transcripts" / "step_up.jsonl"
ANNOTATIONS = DATA_DIR / "teacher_annotations" / "step_up_annotations.jsonl"

## transcripts/step_up.jsonl

In [20]:
transcripts = []
with open(TRANSCRIPTS) as f:
    for line in f:
        line = line.strip()
        if line:
            transcripts.append(json.loads(line))

print(f"Total records: {len(transcripts):,}")

Total records: 21,445


In [21]:
# Records per batch
batch_counts = Counter(r["batch"] for r in transcripts)
print("Records per batch:")
for batch in sorted(batch_counts):
    print(f"  {batch}: {batch_counts[batch]:,}")

Records per batch:
  2025-10-16: 462
  2026-02-13: 109
  2026-03-24: 10,878
  2026-04-27: 9,996


In [22]:
# Turns per session
turn_counts = [len(r["turns"]) for r in transcripts]
avg_turns = sum(turn_counts) / len(turn_counts)
print(f"Turns per session — avg: {avg_turns:.0f}, min: {min(turn_counts)}, max: {max(turn_counts)}")

Turns per session — avg: 349, min: 5, max: 1789


In [23]:
# has_video distribution
video_counts = Counter(r["has_video"] for r in transcripts)
print(f"has_video=True:  {video_counts[True]:,} ({100*video_counts[True]/len(transcripts):.1f}%)")
print(f"has_video=False: {video_counts[False]:,} ({100*video_counts[False]/len(transcripts):.1f}%)")

has_video=True:  20,983 (97.8%)
has_video=False: 462 (2.2%)


In [24]:
# Session duration distribution (in minutes)
durations = []
for r in transcripts:
    turns = r["turns"]
    if turns:
        duration_sec = turns[-1]["end_seconds"] - turns[0]["start_seconds"]
        durations.append(duration_sec / 60)

avg_dur = sum(durations) / len(durations)
print(f"Session duration (minutes) — avg: {avg_dur:.1f}, min: {min(durations):.1f}, max: {max(durations):.1f}")

Session duration (minutes) — avg: 52.8, min: 0.9, max: 180.4


In [25]:
# Unique tutors and students
tutor_ids = {r["session"]["tutor_id"] for r in transcripts}
student_ids = {r["session"]["student_id"] for r in transcripts}
print(f"Unique tutors:  {len(tutor_ids):,}")
print(f"Unique students: {len(student_ids):,}")

Unique tutors:  1,116
Unique students: 1,302


## teacher_annotations/step_up_annotations.jsonl

In [26]:
annotations = []
with open(ANNOTATIONS) as f:
    for line in f:
        line = line.strip()
        if line:
            annotations.append(json.loads(line))

print(f"Total records: {len(annotations):,}")

Total records: 1,095


In [27]:
# Records per annotation type
type_counts = Counter(r["annotation_type"] for r in annotations)
print("Records per annotation_type:")
for atype in sorted(type_counts):
    print(f"  {atype}: {type_counts[atype]}")

Records per annotation_type:
  caption: 79
  rapport: 705
  scaffolding: 311


In [28]:
# Records per batch
ann_batch_counts = Counter(r["batch"] for r in annotations)
print("Records per batch:")
for batch in sorted(ann_batch_counts):
    print(f"  {batch}: {ann_batch_counts[batch]}")

Records per batch:
  2025-10-16: 1095


In [29]:
# Null turn_number_start/end in turn_annotations, by type
for atype in ("rapport", "scaffolding"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_ta = sum(len(r["turn_annotations"]) for r in records)
    null_ta = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is None and ta.get("turn_number_end") is None
    )
    pct = 100 * null_ta / total_ta if total_ta else 0
    print(f"{atype}: {null_ta} / {total_ta} null-anchored turn_annotations ({pct:.1f}%)")

rapport: 23 / 4936 null-anchored turn_annotations (0.5%)
scaffolding: 15 / 1632 null-anchored turn_annotations (0.9%)


In [30]:
# Annotated sessions (unique transcript_ids with scaffolding or rapport)
annotated_ids = {r["transcript_id"] for r in annotations if r["annotation_type"] in ("scaffolding", "rapport")}
print(f"Sessions with scaffolding or rapport annotations: {len(annotated_ids)}")

# Moments per type (excluding null-anchored)
for atype in ("scaffolding", "rapport"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    moments = [
        ta
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is not None
    ]
    print(f"{atype} moments (anchored): {len(moments)}")

Sessions with scaffolding or rapport annotations: 209
scaffolding moments (anchored): 1617
rapport moments (anchored): 4913


## Train/Test Split Statistics

In [31]:
import sys
sys.path.insert(0, str(Path(".").resolve().parent))  # repo root
from annotator.core.utils import compute_iou

SPLIT_FILE = DATA_DIR / "split.json"
GT_DIR = DATA_DIR / "ground_truth_v2"

with open(SPLIT_FILE) as f:
    split = json.load(f)

def load_split_moments(conv_ids):
    moments = []
    for conv_id in conv_ids:
        path = GT_DIR / f"{conv_id}.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for m in data.get("key_moments", []):
            moments.append({**m, "conversation_id": conv_id})
    return moments

train_moments = load_split_moments(split["train"])
test_moments = load_split_moments(split["test"])
print(f"Train: {len(split['train'])} conversations, {len(train_moments)} moments")
print(f"Test:  {len(split['test'])} conversations, {len(test_moments)} moments")

Train: 102 conversations, 3506 moments
Test:  105 conversations, 2953 moments


In [32]:
def split_stats(moments, conv_ids, label):
    total = len(moments)
    print(f"{'='*50}")
    print(f"{label}  ({len(conv_ids)} conversations, {total} moments)")
    print(f"{'='*50}")

    # Annotation type breakdown with per-type label distribution
    type_counts = Counter(m["annotation_type"] for m in moments)
    print("\nAnnotation type and strategy label distribution:")
    for atype in ("scaffolding", "rapport"):
        n = type_counts[atype]
        tc = Counter(m["strategy_label"] for m in moments if m["annotation_type"] == atype)
        print(f"  {atype}: {n}")
        for lbl in ("effective", "partial", "ineffective"):
            print(f"    {lbl}: {tc[lbl]} ({100*tc[lbl]/n:.1f}%)")

    # Count moments with exact turn_start/turn_end match from a different annotator
    # within the same conversation and annotation type.
    by_conv = defaultdict(list)
    for i, m in enumerate(moments):
        by_conv[m["conversation_id"]].append((i, m))

    exact_match_indices = set()
    for conv_moments in by_conv.values():
        for idx_a, (gi_a, m1) in enumerate(conv_moments):
            for gi_b, m2 in conv_moments[idx_a + 1:]:
                if m1["annotator_id"] == m2["annotator_id"]:
                    continue
                if m1["annotation_type"] != m2["annotation_type"]:
                    continue
                if m1["turn_start"] == m2["turn_start"] and m1["turn_end"] == m2["turn_end"]:
                    exact_match_indices.add(gi_a)
                    exact_match_indices.add(gi_b)

    print("\nMoments with matching start/end across >1 annotator:")
    for atype in ("scaffolding", "rapport"):
        type_indices = {i for i, m in enumerate(moments) if m["annotation_type"] == atype}
        n_total = len(type_indices)
        n_match = len(exact_match_indices & type_indices)
        print(f"  {atype}: {n_match} ({100*n_match/n_total:.1f}%)")

split_stats(train_moments, split["train"], "TRAIN")
print()
split_stats(test_moments, split["test"], "TEST")

TRAIN  (102 conversations, 3506 moments)

Annotation type and strategy label distribution:
  scaffolding: 911
    effective: 318 (34.9%)
    partial: 208 (22.8%)
    ineffective: 385 (42.3%)
  rapport: 2595
    effective: 1366 (52.6%)
    partial: 597 (23.0%)
    ineffective: 632 (24.4%)

Moments with matching start/end across >1 annotator:
  scaffolding: 57 (6.3%)
  rapport: 2582 (99.5%)

TEST  (105 conversations, 2953 moments)

Annotation type and strategy label distribution:
  scaffolding: 701
    effective: 245 (35.0%)
    partial: 141 (20.1%)
    ineffective: 315 (44.9%)
  rapport: 2252
    effective: 1142 (50.7%)
    partial: 544 (24.2%)
    ineffective: 566 (25.1%)

Moments with matching start/end across >1 annotator:
  scaffolding: 35 (5.0%)
  rapport: 2224 (98.8%)
